- https://github.com/SWE-bench/SWE-bench
- https://huggingface.co/datasets/princeton-nlp/SWE-bench_Verified
    - https://openai.com/index/introducing-swe-bench-verified/
    - https://www.swebench.com/verified.html

### Env

> SWE-bench 的评测本质是：把模型生成的补丁（patch）应用到真实 repo，再运行 repo 的测试来验证问题是否解决；并且整个过程在 Docker 里做以保证一致性。
- Docker 镜像（image）：可复用的“模板/快照”。SWE-bench 会为不同语言、不同仓库版本构建不同镜像。
    - base → env → instance: 分层构建
- Docker 容器（container）：镜像启动后的“运行实例”。每个评测样本（instance）会创建一个容器来跑测试。
- instance：数据集里的一条任务（某 repo 的某个 issue / commit 场景）。
- TestSpec：harness 把数据集 instance 转成的一份“可执行规格说明”，里面包含：要用的镜像、要写入的脚本、测试集合（FAIL_TO_PASS / PASS_TO_PASS）等；核心构造函数是 make_test_spec()。

#### patch diff

```python
# calculator.py

def divide(a, b):
    return a / b
```
- issue: `When b is 0, divide should return None instead of crashing.`
- diff patch (model generated)
```patch
diff --git a/calculator.py b/calculator.py
index 1234567..89abcde 100644
--- a/calculator.py
+++ b/calculator.py
@@ -1,2 +1,4 @@
 def divide(a, b):
+    if b == 0:
+        return None
     return a / b
```

### swe-agent

```shell
python run.py \
  --model_name "my-coding-model" \
  --api_base "http://localhost:8000/v1" \
  --api_key "EMPTY" \
  --data_path princeton-nlp/SWE-bench_Lite \
  --config_file config/default.yaml
```
- SWE-agent 跑完后，会在它的 trajectories/ 目录下自动为你聚合生成一个格式完美的 predictions.jsonl，你可以直接拿去给 Harness 跑评估。
- github
    - https://github.com/SWE-agent/mini-swe-agent
    - https://github.com/OpenAutoCoder/Agentless

-  ACI（Agent-Computer Interface，智能体-计算机接口）
-  general tools
    -  bash
    -  registry
    -  str_replace_editor
    -  submit / review-on-submit

### evaluation pipeline

```
python -m swebench.harness.run_evaluation \
    --predictions_path path/to/your/predictions.jsonl \
    --max_workers 4 \
    --run_id my_llm_eval_run \
    --dataset_name princeton-nlp/SWE-bench_Lite
```
- predictions → TestSpec → 容器执行 → 解析日志 → report